In [ ]:
#calculate the most porbable path using dijkstra's algorithm for T units

#dijkstra_shortest_path=dijkstra_algorithems(CP, start=2)
#print(dijkstra_shortest_path)
start =15
T=100
start0=start
path=[]
path.append(start)
#edges=dijkstra_algorithems(CP, start)
#print(edges)
for i in range(T):
    if i%100==0:
        print(f"it = {i}")
    filtered=[]
    edges=dijkstra_algorithems(A.CPT, start)
    #print(edges)
    for e in edges:
        if e[2] != 0 and e[2] !=np.inf and (e[1]==start or e[0]==start):
            filtered.append(e)
    #print(filtered)
    #best_edge = min(filtered, key=lambda x: x[2])
    weights=[e[2] for e in filtered]
    #print(weights)
    weights=np.exp(-np.array(weights))
    sumW= sum(weights)
    #print(sumW)
    if sumW==0:
        break
    weights=weights/sumW
    #print(weights)
    #invweights=[]
    #invweights = 1 / np.array(weights)

    proabilities=weights#invweights/sum(invweights)
    #print(proabilities)

    choice = np.random.choice(len(proabilities), p=proabilities)
    best_edge = filtered[choice]
    #print(best_edge)
    if best_edge[1] == start:
        start = best_edge[0]
    else:
        start = best_edge[1]
    path.append(start)
    if start ==-1:
        path.pop() 
        break
print(path)
'''
for i in path:
    plt.scatter(
        DE[A.CPT[0] == i, 1],
        DE[A.CPT[0] == i, 0],
        s=5,    
        #color=ci[i],
    )
#print(cluster_transition_matrix(CP))
plt.xlabel("Energy")
plt.ylabel("Dissipation")
plt.ylim(DE[:, 0].min() - 0.1, DE[:, 0].max() + 0.1)
plt.xlim(DE[:, 1].min() - 0.1, DE[:, 1].max() + 0.1)
plt.title(f"Dijkstra's shortest path from cluster {start0}")
plt.show()
'''

In [ ]:


fig, ax = plt.subplots()

ax.set_xlim(DE[:, 1].min(), DE[:, 1].max())
ax.set_ylim(DE[:, 0].min(), DE[:, 0].max())

# --------------------------------------------------
# PRECOMPUTE cluster indices (FAST instead of masks each frame)
# --------------------------------------------------
clusters = np.unique(CP)
cluster_idx = {c: np.where(CP == c)[0] for c in clusters}

# map each cluster → points
cluster_xy = {
    c: DE[idx][:, [1, 0]] for c, idx in cluster_idx.items()
}

# --------------------------------------------------
# PRECOMPUTE cumulative frames (FAST union via indices)
# --------------------------------------------------
visited = set()
frames_points = []
frames_colors = []

for c in path:
    visited.add(c)

    idx_all = np.concatenate([cluster_idx[v] for v in visited])

    pts = DE[idx_all][:, [1, 0]]
    frames_points.append(pts)

# store cluster per point per frame (for coloring)
for c in path:
    visited.add(c)

    idx_all = np.concatenate([cluster_idx[v] for v in visited])

    frame_colors = np.array(colors)[idx_all]

    # highlight current cluster in red
    current_idx = cluster_idx[c]
    # mark which indices are in current frame selection
    mask = np.isin(idx_all, current_idx)

    frame_colors[mask] = (0.839, 0.152, 0.156, 1.0)

    frames_colors.append(frame_colors)

# --------------------------------------------------
# SCATTER INIT
# --------------------------------------------------
scat = ax.scatter([], [], s=5)

# --------------------------------------------------
# UPDATE (VERY FAST)
# --------------------------------------------------
def update(i):
    scat.set_offsets(frames_points[i])
    scat.set_color(frames_colors[i])
    ax.set_title(f"Step {i} | Cluster {path[i]}")
    return scat,

# --------------------------------------------------
# ANIMATION
# --------------------------------------------------
ani = FuncAnimation(
    fig,
    update,
    frames=len(path),
    interval=300,
    blit=False
)

# --------------------------------------------------
# SAVE (FAST)
# --------------------------------------------------
writer = FFMpegWriter(fps=2)
ani.save("DE.mp4", writer=writer)

plt.show()

In [ ]:
#K-means Clustering 
if __name__=='__main__':
    
    dt,N,istart = 0.05,510000,10000       # time-step and number of trajectory points
    Re=800
    D,E,a = MFE_Sequence(dt,N,istart,Re,2*np.pi,2*np.pi)   # generate data sequence 
    nclust = 10                # number of clusters ('boxes')  
    
    plt.figure(1)             # visualization 
    plt.ion()
    plt.clf()
    plt.plot(E,D)
    plt.gca().set_aspect(0.1)
    plt.show()
    plt.pause(0.001)

    DE = np.vstack((D,E)).T
    print('clustering in phase space') 

    km = KMeans(n_clusters=nclust, init='k-means++').fit(DE)  # 'discretize' into clusters 
    C  = km.labels_                     # cluster membership
    print(C.shape) 
    CC = km.cluster_centers_            # cluster centroids 
    c  = cm.tab10(np.linspace(0, 1, nclust))  # color coding
    cluster_counts = np.bincount(C, minlength=nclust)
    print(f"The minimum number of snapshots per cluster = {min(cluster_counts)}")
    colors = [c[label % len(c)] for label in C]
    for i in range(len(CC)):
        ii = np.where(C==i)[0]
        DD,EE = D[ii],E[ii]
    plt.scatter(E,D,s=5,c=colors)
    plt.show()
    print(CC)

In [ ]:

'''
#Reorder clusters based on the time series 
# Relabel clusters by order of first appearance in trajectory
CC=clusterCentroids(CP,DE)  # Recompute cluster centers after reclustering
seen_clusters = {}  # maps old label → new label
new_label_counter = 0

for i in range(len(CP)):
    old_label = CP[i]
    if old_label not in seen_clusters:
        seen_clusters[old_label] = new_label_counter
        new_label_counter += 1

# Apply relabeling
relabel_map = seen_clusters
CP = np.array([relabel_map[label] for label in CP])
# Reorder cluster centers to match new labeling
CC_new = np.zeros_like(CC)
for old_label, new_label in relabel_map.items():
    CC_new[new_label] = CC[old_label]
CC = CC_new
'''

In [ ]:
def scaled_ulamGalerkin(Data,Nd=5,dt=0.05,sim=np.inf,sample=1):        #Data here are of size (Time x physical variables)(Rows are time while coulmns are physical variables)

    x=Data.copy()      
    mini = np.min(x, axis=0)        #Extract minimum value of all physical variables 
    maxi = np.max(x, axis=0)        #Extract maximum value of all physical variables
    delta=(maxi+10e-12-mini)/Nd            #Discretize the state space into cells
    idx=np.floor((x-mini)/delta).astype(int) #Assign each point in the data to a cell in the state space
    unique_rows, inverse_indices = np.unique(idx, axis=0, return_inverse=True)  #unique_rows of x = boxnumbers or box id, inverse_indices are snapshots id 
    bx_to_p_dict={}
    p_to_bx_dict=np.array([tuple(col) for col in idx])                          #This a list of boxes of each snapshot p_to_bx_dict[0]=[1,1]
    #M[0,0]=1                                                                  #first snapshot assignment
    for i in range(len(unique_rows)):
        bx_to_p_dict[tuple(unique_rows[i])]= np.where(inverse_indices == i)[0]
    
    M=np.zeros((len(unique_rows),len(unique_rows)))     #start the markov matrix
    U_local=np.zeros(len(unique_rows))
    if np.isfinite(sim):
        boundary = set(np.arange(sim, len(inverse_indices), sample))
        #print(boundary)
    else:
        boundary = set()
    for i in range(len(inverse_indices)-1):
        if i+1 in boundary:
            continue
        M[inverse_indices[i+1],inverse_indices[i]]=M[inverse_indices[i+1],inverse_indices[i]]+1
        if inverse_indices[i+1]==inverse_indices[i]:
            U_local[inverse_indices[i]]+=np.linalg.norm(x[i+1]-x[i])
    U_local=U_local/M.sum(axis=0)
    no_self_transition_idx=U_local==0
    dt_U_local_inv_dx=U_local*(dt/np.linalg.norm(delta))
    dt_U_local_inv_dx[no_self_transition_idx]=1
    idx = np.diag_indices_from(M)
    M[idx] *= dt_U_local_inv_dx
    M=np.floor(M)
    M = M / M.sum(axis=0,keepdims=True)  #Normalize over the rows where sum of columns equal to 1 (j>>>all i)
    return M   #Return markov matrix, a dictionary of box:[snapshots,,,], a list of snapshot assignment to boxes P:[box,,,,]
print(scaled_ulamGalerkin(DE,30,dt)[:,0])
